# Optimize Random Forest Hyperparameters

This notebook optimizes **Random Forest** hyperparameters using **GridSearchCV** and **RandomizedSearchCV**, comparing configurations via nested cross-validation.

**Context from previous evaluation notebooks:**
- `Spot_Checking_Balancing.ipynb` -> best balancing strategy (average across models): **No Balancing**
- `Spot_Checking_DimensionalityReduction.ipynb` -> best DR technique (average across models): **PCA - 90% variance**
- `Spot_Checking_ModelComparison.ipynb` -> best model: **Random Forest**
- `Evaluate_RF_Balancing.ipynb` -> best balancing strategy for RF: **No Balancing**
- `Evaluate_RF_DimensionalityReduction.ipynb` -> best DR technique for RF: **PCA - 90% variance**

**Goal:** Find the optimal hyperparameter configuration for Random Forest, comparing exhaustive (GridSearchCV) and stochastic (RandomizedSearchCV) search strategies.

**Data flow:**
- Input: `kidney_disease_encoded.csv` -> dataset already encoded (`Encode_Categorical_Values.ipynb`)
- `MinMaxScaler` fitted only on training data of each fold (no data leakage)
- No balancing applied (confirmed best strategy for RF in `Evaluate_RF_Balancing.ipynb`)
- No dimensionality reduction (RF evaluated with full feature set as baseline for tuning)

**Evaluation approach:**
- **Nested cross-validation**: outer `StratifiedKFold(k=5)` for unbiased performance estimation; inner `StratifiedKFold(k=3)` for hyperparameter selection
- Out-of-fold predictions are collected in the main loop - confusion matrices and ROC curves reuse these, avoiding redundant fits
- Metrics: F1, Accuracy, Precision, Recall
- Visualizations: summary table, boxplots, confusion matrices, ROC curves, ranking

**Search configurations evaluated:**

| Configuration | Strategy | Description |
|---|---|---|
| Baseline | — | Default `RandomForestClassifier` params (reference) |
| GridSearchCV — focused grid | Exhaustive | Predefined grid; every combination evaluated |
| RandomizedSearchCV — broad (n=50) | Stochastic | Wide distributions; 50 random samples |
| RandomizedSearchCV — fine-tuned (n=30) | Stochastic | Narrowed distributions; 30 random samples |

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc,
)
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint

# imblearn Pipeline supports samplers (kept for consistency with other notebooks)
from imblearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

## 2. Load and Prepare Data

In [ ]:
df = pd.read_csv('../data/processed/kidney_disease_encoded.csv')

print(f'Shape: {df.shape}')
print(f'\nTarget variable distribution:')
print(df['class'].value_counts())

In [ ]:
X = df.drop(columns=['class'])
y = LabelEncoder().fit_transform(df['class'])  # ckd=0, notckd=1 (or vice versa)

target_names = df['class'].unique().tolist()  # original labels preserved for display

print(f'Features: {X.shape[1]} columns, {X.shape[0]} samples')
print(f'Encoded classes: {np.unique(y)} (original: {df["class"].unique()})')

## 3. Reusable Pipeline

`make_pipeline` encapsulates the full preprocessing + classifier chain:

1. **`MinMaxScaler`:** fitted only on training data of each fold (no data leakage)
2. **`classifier`:** estimator to evaluate

No balancing and no dimensionality reduction are applied, consistent with the best strategy confirmed for RF. The hyperparameter search wraps this pipeline, varying only the classifier parameters.

In [ ]:
def make_pipeline(classifier):
    """
    Build a reusable imblearn Pipeline for hyperparameter optimization.

    Parameters
    ----------
    classifier : sklearn estimator
        The classification model.

    Returns
    -------
    imblearn.pipeline.Pipeline
        No balancing (best strategy for RF: No Balancing).
        No dimensionality reduction (RF evaluated with full feature set).
    """
    steps = [
        ('scaler', MinMaxScaler()),
        ('classifier', classifier),
    ]
    return Pipeline(steps)

## 4. Search Configurations

Each configuration defines a **search strategy** (Grid or Random) and the **parameter space** to explore.
Pipeline parameter names follow the `stepname__param` convention required by sklearn pipelines.

> **GridSearchCV — focused grid:** evaluates every combination in the predefined grid (exhaustive).
> 
> **RandomizedSearchCV — broad:** samples 50 random combinations from wide distributions (exploration).
>
> **RandomizedSearchCV — fine-tuned:** samples 30 combinations from narrowed distributions around promising regions (exploitation).

In [ ]:
search_configs = {
    # ---- Reference -------------------------------------------------------
    'Baseline (default params)': {
        'search_type': None,
    },

    # ---- Exhaustive search -----------------------------------------------
    'GridSearchCV — focused grid': {
        'search_type': 'grid',
        'param_grid': {
            'classifier__n_estimators':      [100, 200, 300],
            'classifier__max_depth':         [None, 10, 20],
            'classifier__min_samples_split': [2, 5, 10],
            'classifier__min_samples_leaf':  [1, 2, 4],
            'classifier__max_features':      ['sqrt', 'log2'],
        },
    },

    # ---- Stochastic search — broad exploration ---------------------------
    'RandomizedSearchCV — broad (n=50)': {
        'search_type': 'random',
        'param_distributions': {
            'classifier__n_estimators':      randint(50, 500),
            'classifier__max_depth':         [None, 5, 10, 15, 20, 30],
            'classifier__min_samples_split': randint(2, 20),
            'classifier__min_samples_leaf':  randint(1, 10),
            'classifier__max_features':      ['sqrt', 'log2', None],
        },
        'n_iter': 50,
    },

    # ---- Stochastic search — fine exploitation ---------------------------
    'RandomizedSearchCV — fine-tuned (n=30)': {
        'search_type': 'random',
        'param_distributions': {
            'classifier__n_estimators':      randint(150, 400),
            'classifier__max_depth':         [None, 10, 15, 20],
            'classifier__min_samples_split': randint(2, 8),
            'classifier__min_samples_leaf':  [1, 2, 3],
            'classifier__max_features':      ['sqrt'],
        },
        'n_iter': 30,
    },
}

# Summary of search space sizes
print('Search configurations defined:')
for name, cfg in search_configs.items():
    st = cfg['search_type']
    if st is None:
        print(f'  [{name}]: no search (default params)')
    elif st == 'grid':
        n_comb = 1
        for vals in cfg['param_grid'].values():
            n_comb *= len(vals)
        print(f'  [{name}]: GridSearchCV - {n_comb} combinations × 3 inner folds = {n_comb * 3} fits/outer fold')
    elif st == 'random':
        n_iter = cfg['n_iter']
        print(f'  [{name}]: RandomizedSearchCV - {n_iter} iterations × 3 inner folds = {n_iter * 3} fits/outer fold')

## 5. Nested Cross-Validation — Performance Evaluation

**Nested CV design:**
- **Outer loop** (`StratifiedKFold, k=5`): unbiased estimate of generalisation performance
- **Inner loop** (`StratifiedKFold, k=3`): hyperparameter selection inside each outer training fold

Out-of-fold predictions (`y_pred_oof`, `y_proba_oof`) are collected in the same loop to avoid redundant computation in later sections (confusion matrices, ROC curves).

> **Note on computation time:** GridSearchCV with 162 combinations × 3 inner × 5 outer folds = 2430 RF fits. RandomizedSearchCV with 50 iterations has 750 fits. On the kidney disease dataset (~199 samples), this runs in seconds to a few minutes.

In [ ]:
def run_nested_cv(config_name, config, X, y, outer_cv, inner_cv):
    """
    Run nested cross-validation for a single search configuration.

    Parameters
    ----------
    config_name : str
    config : dict
        Must contain 'search_type' and optionally 'param_grid' / 'param_distributions' / 'n_iter'.
    X : pd.DataFrame
    y : np.ndarray
    outer_cv : StratifiedKFold
    inner_cv : StratifiedKFold

    Returns
    -------
    fold_metrics : dict
        Per-fold metric values for accuracy, precision, recall, f1.
    y_pred_oof : np.ndarray
        Out-of-fold class predictions (shape = n_samples).
    y_proba_oof : np.ndarray
        Out-of-fold predicted probabilities (shape = (n_samples, 2)).
    best_params_list : list of dict
        Best hyperparameters from each outer fold (empty list for baseline).
    """
    fold_metrics     = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
    y_pred_oof       = np.empty(len(y), dtype=int)
    y_proba_oof      = np.empty((len(y), 2))
    best_params_list = []

    for train_idx, test_idx in outer_cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        base_pipeline = make_pipeline(RandomForestClassifier(random_state=42))
        search_type   = config.get('search_type')

        if search_type is None:
            estimator = base_pipeline

        elif search_type == 'grid':
            estimator = GridSearchCV(
                estimator  = base_pipeline,
                param_grid = config['param_grid'],
                cv         = inner_cv,
                scoring    = 'f1',
                n_jobs     = -1,
                refit      = True,
            )

        elif search_type == 'random':
            estimator = RandomizedSearchCV(
                estimator            = base_pipeline,
                param_distributions  = config['param_distributions'],
                n_iter               = config['n_iter'],
                cv                   = inner_cv,
                scoring              = 'f1',
                random_state         = 42,
                n_jobs               = -1,
                refit                = True,
            )

        estimator.fit(X_train, y_train)

        y_pred_oof[test_idx]  = estimator.predict(X_test)
        y_proba_oof[test_idx] = estimator.predict_proba(X_test)

        fold_metrics['accuracy'].append(accuracy_score(y_test, y_pred_oof[test_idx]))
        fold_metrics['precision'].append(precision_score(y_test, y_pred_oof[test_idx]))
        fold_metrics['recall'].append(recall_score(y_test, y_pred_oof[test_idx]))
        fold_metrics['f1'].append(f1_score(y_test, y_pred_oof[test_idx]))

        if hasattr(estimator, 'best_params_'):
            best_params_list.append(estimator.best_params_)

    return fold_metrics, y_pred_oof, y_proba_oof, best_params_list


outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# results[config_name] = {fold_metrics, y_pred_oof, y_proba_oof, best_params_list}
results = {}

for config_name, config in search_configs.items():
    print(f'Running: {config_name} ...')
    fold_metrics, y_pred_oof, y_proba_oof, best_params_list = run_nested_cv(
        config_name, config, X, y, outer_cv, inner_cv
    )
    results[config_name] = {
        'fold_metrics':     fold_metrics,
        'y_pred_oof':       y_pred_oof,
        'y_proba_oof':      y_proba_oof,
        'best_params_list': best_params_list,
    }
    f1_mean = np.mean(fold_metrics['f1'])
    f1_std  = np.std(fold_metrics['f1'])
    print(f'  F1: {f1_mean:.4f} ± {f1_std:.4f}')
    print()

print('All done.')

## 6. Summary Table

In [ ]:
scoring = ['accuracy', 'precision', 'recall', 'f1']

rows = []
for config_name, res in results.items():
    row = {'Configuration': config_name}
    for metric in scoring:
        values = res['fold_metrics'][metric]
        row[f'{metric}_mean'] = np.mean(values)
        row[f'{metric}_std']  = np.std(values)
    rows.append(row)

summary_df = (
    pd.DataFrame(rows)
    .sort_values('f1_mean', ascending=False)
    .set_index('Configuration')
)

float_cols = summary_df.select_dtypes(include='float').columns.tolist()
mean_cols  = [c for c in float_cols if c.endswith('_mean')]

display(
    summary_df.style
    .format('{:.4f}', subset=float_cols)
    .background_gradient(cmap='RdYlGn', axis=0, subset=mean_cols)
)

## 7. F1-Score Boxplot (CV Folds)

Each box represents the distribution of F1 scores across the 5 outer folds. A narrower, higher box indicates a configuration that is both accurate and consistent.

In [ ]:
f1_data = {
    config_name: res['fold_metrics']['f1']
    for config_name, res in results.items()
}

fig, ax = plt.subplots(figsize=(12, 5))
pd.DataFrame(f1_data).boxplot(ax=ax)
ax.set_title('Random Forest — F1-score Distribution per Search Configuration (Nested k=5/3 CV)')
ax.set_ylabel('F1-score')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 8. Confusion Matrices (Aggregated over CV Folds)

Out-of-fold predictions collected during nested CV — aggregates class predictions from all 5 outer folds, giving one confusion matrix per configuration over the full dataset.

In [ ]:
n_configs = len(results)
ncols = 2
nrows = int(np.ceil(n_configs / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes = axes.flatten()

for ax, (config_name, res) in zip(axes, results.items()):
    cm   = confusion_matrix(y, res['y_pred_oof'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(config_name, fontsize=9)

for ax in axes[n_configs:]:
    ax.set_visible(False)

plt.suptitle('Random Forest — Confusion Matrices per Search Configuration', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9. ROC Curves

Out-of-fold probability predictions collected during nested CV. AUC is computed over the aggregated out-of-fold predictions (one curve per configuration).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for config_name, res in results.items():
    fpr, tpr, _ = roc_curve(y, res['y_proba_oof'][:, 1])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{config_name} (AUC = {roc_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Random Forest — ROC Curves per Search Configuration')
ax.legend(loc='lower right', fontsize=8)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.show()

## 10. Best Hyperparameters Analysis

The tables below show which hyperparameter values were selected in each outer fold. Stability across folds (same value chosen repeatedly) indicates a robust optimum.

**Section 10a** — per-fold values (from nested CV)  
**Section 10b** — final best params from a single full-dataset search (for analysis/reporting only; performance is estimated via nested CV above)

In [ ]:
# --- 10a. Per-fold best params from nested CV ---
print('=' * 72)
print('BEST HYPERPARAMETERS PER OUTER FOLD (from nested CV)')
print('=' * 72)

for config_name, res in results.items():
    bp_list = res['best_params_list']
    if not bp_list:
        print(f'\n[{config_name}]')
        print('  No search performed — using default RandomForestClassifier params.')
        continue

    print(f'\n[{config_name}]')
    param_keys = list(bp_list[0].keys())
    fold_labels = [f'Fold {i+1}' for i in range(len(bp_list))]

    param_table = pd.DataFrame(bp_list, index=fold_labels).T
    param_table.index = [k.replace('classifier__', '') for k in param_table.index]
    display(param_table)

In [ ]:
# --- 10b. Final best params from full-dataset search (analysis only) ---
print('=' * 72)
print('BEST HYPERPARAMETERS — FIT ON FULL DATASET (for analysis/reporting only)')
print('Performance estimates come from nested CV above, not from this fit.')
print('=' * 72)

final_best_params = {}
full_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for config_name, config in search_configs.items():
    search_type = config.get('search_type')

    if search_type is None:
        print(f'\n[{config_name}]: using default RandomForestClassifier params (no search).')
        final_best_params[config_name] = {}
        continue

    base_pipeline = make_pipeline(RandomForestClassifier(random_state=42))

    if search_type == 'grid':
        search = GridSearchCV(
            estimator  = base_pipeline,
            param_grid = config['param_grid'],
            cv         = full_cv,
            scoring    = 'f1',
            n_jobs     = -1,
            refit      = True,
        )
    elif search_type == 'random':
        search = RandomizedSearchCV(
            estimator           = base_pipeline,
            param_distributions = config['param_distributions'],
            n_iter              = config['n_iter'],
            cv                  = full_cv,
            scoring             = 'f1',
            random_state        = 42,
            n_jobs              = -1,
            refit               = True,
        )

    search.fit(X, y)
    final_best_params[config_name] = search.best_params_

    best_p = {k.replace('classifier__', ''): v for k, v in search.best_params_.items()}
    print(f'\n[{config_name}]')
    print(f'  Best CV score (F1): {search.best_score_:.4f}')
    for k, v in best_p.items():
        print(f'  {k}: {v}')

## 11. Ranking

Primary criterion: highest mean F1.  
Tie-breaking: lower std (more consistent across outer folds).

In [ ]:
ranking_df = (
    summary_df[['f1_mean', 'f1_std', 'accuracy_mean', 'precision_mean', 'recall_mean']]
    .sort_values(['f1_mean', 'f1_std'], ascending=[False, True])
    .reset_index()
)

ranking_df.index      = ranking_df.index + 1
ranking_df.index.name = 'Rank'

float_cols = ranking_df.select_dtypes(include='float').columns.tolist()
display(
    ranking_df.style
    .format('{:.4f}', subset=float_cols)
    .background_gradient(cmap='RdYlGn', axis=0, subset=float_cols)
)